# Descripción y análisis exploratorio de datos

**Materia:** Análisis de Series de Tiempo y Pronósticos (1C-2026)

**Grupo:** 9

**Integrantes:**
- Lucas Achaval - Email: lachavalrodriguez@estudiantes.unsam.edu.ar
- Marcos Achaval - Email: machavalrodriguez@unsam-bue.edu.ar

**Enlace al conjunto de datos original:** Dataset propio de estación meteorológica de club náutico en Potrerillos (archivo local: data/station_15338.csv).

## Descripción breve del conjunto de datos y su origen

Este conjunto de datos proviene de una estación meteorológica instalada en un club náutico en Potrerillos, Mendoza. La estación registra una observación por minuto con variables de viento (promedio y máximo), dirección, temperatura, humedad relativa y presión atmosférica.

El archivo contiene datos desde 2025-08-01 hasta 2026-04-14, y representa mediciones reales en un entorno de montaña donde el viento térmico tiene un comportamiento diario marcado.

## Problema o pregunta a resolver

Potrerillos, y en particular la zona del embalse, presenta un comportamiento del viento y del clima muy característico. Se trata de un entorno montañoso donde predominan los vientos térmicos, es decir, vientos condicionados por diferencias de temperatura a lo largo del día.

Durante el verano, el régimen de viento suele seguir un patrón muy definido: las intensidades son bajas durante la madrugada, aumentan progresivamente por la mañana, alcanzan valores más estables al mediodía y luego comienzan a disminuir por la tarde y la noche. La dirección del viento también responde a una dinámica clara en esta época del año. En general, sopla desde el sector sudeste (135°).

Sin embargo, a pesar de este patrón general, los pronósticos de viento para esta zona suelen fallar en la representación de la intensidad observada. Con frecuencia subestiman los máximos diarios y no capturan con suficiente precisión la señal térmica local. Por eso, nuestro objetivo con este proyecto es desarrollar un modelo de pronóstico de viento para este lugar que sea más preciso que los existentes y que capture adecuadamente los efectos térmicos. Para ello nos vamos a enfocar específicamente en el pronóstico de la intensidad del viento (variable objetivo: `wind_avg`), y no en su dirección.

Métricas de evaluación propuestas:
- MAE (error absoluto medio)
- RMSE (raíz del error cuadrático medio)
- R2

El horizonte de pronóstico será de 12 horas y se generarán 12 predicciones a futuro, una por cada hora. Para ello, primero se realizará una agregación temporal (resample) del conjunto de datos a resolución horaria mediante la mediana, con el fin de alinear la escala temporal del modelo con el objetivo del pronóstico.

## Análisis exploratorio de datos

### 4.a) ¿Por qué este conjunto debe abordarse como serie de tiempo?

In [1]:
import pandas as pd

df = pd.read_csv("../data/station_15338.csv")
df.set_index(pd.to_datetime(df["datetime"]), inplace=True)
df.drop(columns=["datetime", "unixtime"], inplace=True)
df.head(10)

,wind_avg,wind_max,wind_min,wind_direction,temperature,rh,mslp,gustiness
datetime,,,,,,,,
2025-08-01 00:01:00,8.9,11.3,NaN,61.0,10.0,73.0,855.1,26.0
2025-08-01 00:02:00,8.7,11.3,NaN,73.0,10.0,73.0,855.1,29.0
2025-08-01 00:03:00,12.0,14.8,NaN,57.0,10.0,73.3,855.2,23.0
2025-08-01 00:04:00,8.7,14.8,NaN,61.0,10.0,73.0,855.2,69.0
2025-08-01 00:05:00,7.8,14.8,NaN,73.0,10.0,73.5,855.2,90.0
2025-08-01 00:06:00,7.4,11.6,NaN,67.0,9.9,73.0,855.2,56.0
2025-08-01 00:07:00,8.5,11.3,NaN,70.0,9.9,73.8,855.2,33.0
2025-08-01 00:08:00,10.5,13.4,NaN,81.0,9.9,73.0,855.2,27.0
2025-08-01 00:09:00,11.3,13.4,NaN,89.0,9.9,72.8,855.1,18.0


El conjunto de datos está ordenado cronológicamente y cada observación depende del instante de medición. Por lo tanto, el orden temporal no es intercambiable y es por esto que debemos tratar el problema como una serie de tiempo.

### 4.b) Relación entre cada paso y el tiempo real

In [ ]:
import numpy as np

df["interval"] = np.float32((df.index - df.index[0]).total_seconds() / 60)
df["time_diff"] = df["interval"].diff().fillna(0)

print("Análisis de pasos del tiempo:")
print(
    f"mínimo: {df['time_diff'].min():.2f}, \npromedio: {df['time_diff'].mean():.2f}, \nmediana: {df['time_diff'].median():.2f}, \nmáximo: {df['time_diff'].max():.2f}"
)

print(f"\nValor mínimo de tiempo: {df.index.min()}")
print(f"Valor máximo de tiempo: {df.index.max()}")

Análisis de pasos del tiempo:
mínimo: 0.00, 
promedio: 1.05, 
mediana: 1.00, 
máximo: 5761.00

Valor mínimo de tiempo: 2025-08-01 00:01:00
Valor máximo de tiempo: 2026-04-14 00:00:00


Se observa que el promedio de los pasos temporales es de aproximadamente 1.05 minutos, lo cual indica que en algunos momentos hubo irregularidades (por ejemplo, faltantes o interrupciones de registro). Sin embargo, la mediana del paso es exactamente 1 minuto. Como la mediana es robusta frente a valores atípicos, este resultado sugiere que la frecuencia principal de muestreo es por minuto y que las irregularidades son puntuales.

Además, el índice temporal abarca desde el 1 de agosto de 2025 hasta el 14 de abril de 2026.

### 4.c) Valores faltantes: diagnóstico, completado e impacto

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 350492 entries, 2025-08-01 00:01:00 to 2026-04-14 00:00:00
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   wind_avg        350492 non-null  float64
 1   wind_max        350492 non-null  float64
 2   wind_min        0 non-null       float64
 3   wind_direction  350491 non-null  float64
 4   temperature     350492 non-null  float64
 5   rh              350492 non-null  float64
 6   mslp            350473 non-null  float64
 7   gustiness       346917 non-null  float64
 8   interval        350492 non-null  float32
 9   time_diff       350492 non-null  float32
dtypes: float32(2), float64(8)
memory usage: 26.7 MB


El análisis de completitud muestra valores faltantes en `wind_direction`, `mslp` y `gustiness`, mientras que `wind_min` no contiene datos válidos en todo el período.

Estrategia de tratamiento propuesta:
- `wind_min`: excluir esta variable del modelado, ya que no aporta información utilizable.
- Faltantes puntuales en `wind_direction`, `mslp` y `gustiness`: imputar con métodos temporales (interpolación por tiempo y/o propagación hacia adelante y hacia atrás en huecos cortos).
- Si existen huecos extensos: considerar imputación más conservadora (por ejemplo, mediana por franja horaria).

Impacto en el análisis posterior:
- La imputación reduce pérdida de datos y evita sesgos por eliminación masiva de filas.
- También puede suavizar extremos y disminuir la variabilidad real, afectando métricas de dispersión, autocorrelación y desempeño predictivo.
- Por ello, conviene comparar resultados con y sin imputación, o con distintos esquemas de imputación, para verificar robustez.